In [51]:
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))

from config import config
from domain.ipm import IPM, INDICADORES_ELIMINADOS, CARACTERISTICAS_NUEVAS
from domain.variables_naming import ARCHIVOS, VARIABLES, variables_ipm_dane
from utils.triple_plot import normality_report

# Consolidar Tabla Final

**Asistido por Claude Opus 5.5**

Dado que la parte de modelamiento del paper se realiza sobre un solo año se decidio utilizar solo la ECV correspondiente al 2025 ya que es la mas reciente que se tiene hasta el momento

## Objetivo:

Generar un modelo capaz de clacificar si un hogar es multidimensionalmente pobre, por lo tanto el sujeto de estudio es el hogar y no el individuo.

In [52]:
CSV_DIR = config.EXTRACTED / "ECV_2025_extracted" / "csv"

KEY_COLS = ["DIRECTORIO", "SECUENCIA_ENCUESTA", "SECUENCIA_P", "ORDEN"]
SHORT = {
    "DIRECTORIO": "D",
    "SECUENCIA_ENCUESTA": "SE",
    "SECUENCIA_P": "SP",
    "ORDEN": "ORDEN",
}

DF_SIZES = {}

# combinaciones que se usan a lo largo del notebook
D_SE = ["DIRECTORIO", "SECUENCIA_ENCUESTA"]
D_SP = ["DIRECTORIO", "SECUENCIA_P"]
D_SP_ORDEN = ["DIRECTORIO", "SECUENCIA_P", "ORDEN"]

# alias corto -> nombre del archivo, para no arrastrar tildes en el código
FILES = {
    "vivienda": "Datos de la vivienda.csv",
    "servicios": "Servicios del hogar.csv",
    "tenencia": (
        "Tenencia y financiacion de la vivienda que ocupa el hogar.csv"
    ),
    "condiciones": "Condiciones de vida del hogar y tenencia de bienes.csv",
    "adquisicion": "Adquisición de bienes y servicios.csv",
    "caracteristicas": "Características y composición del hogar.csv",
    "salud": "Salud.csv",
    "diseno": "Variables diseño muestral.csv",
    "educacion": "Educación.csv",
    "tic": "Tecnologías de información y comunicación.csv",
    "fuerza_trabajo": "Fuerza de trabajo.csv",
    "trabajo_infantil": "Trabajo infantil.csv",
    "menores_5": (
        "Atención integral de los niños y niñas menores de 5 años.csv"
    ),
    "item": "Adquisición de bienes y servicios por ítem.CSV",
    "programas": (
        "Condiciones de vida del hogar y tenencia de bienes (programas).csv"
    ),
    "escritura": (
        "Tenencia y financiacion de la vivienda que ocupa el hogar "
        "(personas escritura).csv"
    ),
    "pdet": "FEX_PDET_VIV_2025.csv",
}

# columnas auxiliares que se cargan junto con las llaves
EXTRA_COLS = {
    "servicios": ["CANT_PERSONAS_HOGAR"],
    "item": ["P3204"],
    "escritura": ["P8692S3A1"],
}

In [53]:
def read_data(alias, extra=()):
    """Lee las columnas llave presentes en el archivo, más `extra`."""
    path = CSV_DIR / FILES[alias]
    decimal = "." if alias == "item" else ","
    if alias == "vivienda":
        return pd.read_csv(
            path,
            sep=";",
            decimal=decimal,
            dtype={"P1_DEPARTAMENTO": str, "P1_MUNICIPIO": str},
        )
    return pd.read_csv(path, sep=";", decimal=decimal)

In [54]:
keys = {alias: read_data(alias, EXTRA_COLS.get(alias, ())) for alias in FILES}

In [55]:
keys["vivienda"]

,DIRECTORIO,SECUENCIA_ENCUESTA,SECUENCIA_P,ORDEN,P1_DEPARTAMENTO,P1_MUNICIPIO,REGION,FEX_C,CANT_HOG_COMPLETOS,CANT_HOGARES_VIVIENDA,...,P5661S9,P5661S5,P5661S6,P5661S7,P3157,P3157S1,P3157S2,P3157S3,P3157S4,P3158
0,8369550,1,1,1,13,1,1,472.131901,1,1,...,1,1,1,1,NaN,2,2,2,2,2
1,8369551,1,1,1,13,1,1,308.854319,1,1,...,1,1,2,1,NaN,2,2,2,2,2
2,8369552,1,1,1,13,1,1,428.516239,1,1,...,1,2,2,2,NaN,2,1,1,1,2
3,8369553,1,1,1,13,1,1,268.976656,1,1,...,1,2,3,3,NaN,1,1,1,1,1
4,8369554,1,1,1,13,1,1,353.949442,1,1,...,1,1,2,1,NaN,2,2,2,2,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
86843,8489518,1,1,1,99,524,9,23.804410,1,1,...,1,1,1,1,NaN,2,2,2,2,2
86844,8489519,1,1,1,99,524,9,19.043528,1,1,...,1,1,1,1,NaN,2,2,2,2,2
86845,8489520,1,1,1,99,524,9,19.043528,1,1,...,1,1,1,1,NaN,2,2,2,2,2
86846,8489521,1,1,1,99,524,9,19.043528,1,1,...,1,1,1,1,NaN,2,2,2,2,2


Una vez cargadas todas las tablas de manera individual se procede a consolidar las variables de interes, expuestas en la siguiente tabla, con su uso ya sea corte o modelamiento, y su nombre segun el diseño de la encuesta. Ademas antes de iniciar con el EDA y analisis previos al modelamiento se opta por realizar el Dual Cutoff Method (para crear la variable binaria objetivo) utilizando los cortes que el DANE reporta, adicionalmente se evita que variables que actuen en el corte se solapen con las predictoras. 

## Consolidar DF

La tabla final queda a **nivel persona**: una fila por persona con todas las
variables de la ECV 2025. Los cruces usan las llaves verificadas en
`full_analisis.ipynb`:

| Nivel | Llave en su archivo | Con qué se cruza en la tabla final |
| --- | --- | --- |
| vivienda | `DIRECTORIO` | `DIRECTORIO` |
| hogar | `DIRECTORIO` + `SECUENCIA_ENCUESTA` | `DIRECTORIO` + `SECUENCIA_P` |
| persona | `DIRECTORIO` + `SECUENCIA_P` + `ORDEN` | las mismas tres |

Tres detalles que salen de ese análisis:

- En los archivos de persona `SECUENCIA_ENCUESTA` repite lo que ya dice
  `ORDEN`, así que se descarta y la llave de persona queda `D + SP + ORDEN`.
- El hogar se llama `SECUENCIA_ENCUESTA` en los archivos de hogar y
  `SECUENCIA_P` en los de persona; por eso se renombra antes de cruzar.
- `FEX_C` es la única columna no llave que aparece en los 13 archivos. Se
  conserva solo la de `diseno`, porque `condiciones` y `diseno` difieren del
  resto en 4.553 hogares por redondeo.

Las variables de hogar y de vivienda **se repiten en cada integrante del
hogar**, que es lo que necesita el procesamiento posterior. `item`,
`programas` y `escritura` quedan fuera: traen varias filas por hogar y no
caben en una tabla de una fila por persona sin agregarlas antes.

In [56]:
# archivos que aportan una fila por persona y una fila por hogar
PERSON_FILES = [
    "salud",
    "diseno",
    "educacion",
    "tic",
    "fuerza_trabajo",
    "trabajo_infantil",
    "menores_5",
]
HOUSEHOLD_FILES = ["servicios", "tenencia", "condiciones", "adquisicion"]

In [57]:
# base: una fila por persona. SECUENCIA_ENCUESTA se descarta porque repite
# a ORDEN; FEX_C se conserva solo desde 'diseno'.
df = keys["caracteristicas"].drop(columns=["SECUENCIA_ENCUESTA", "FEX_C"])

for alias in PERSON_FILES:
    drop_cols = ["SECUENCIA_ENCUESTA"]
    if alias != "diseno":
        drop_cols.append("FEX_C")
    right = keys[alias].drop(columns=drop_cols, errors="ignore")
    df = df.merge(right, on=D_SP_ORDEN, how="left", validate="1:1")

print(df.shape)

(235350, 567)


In [58]:
# el hogar es SECUENCIA_ENCUESTA en estos archivos y SECUENCIA_P en la
# tabla de personas: se renombra para cruzar por D + SP
for alias in HOUSEHOLD_FILES:
    right = (
        keys[alias]
        .drop(columns=["SECUENCIA_P", "ORDEN", "FEX_C"])
        .rename(columns={"SECUENCIA_ENCUESTA": "SECUENCIA_P"})
    )
    df = df.merge(right, on=D_SP, how="left", validate="m:1")

print(df.shape)

(235350, 861)


In [59]:
# vivienda: sus SE, SP y ORDEN valen siempre 1, no aportan nada
right = keys["vivienda"].drop(
    columns=["SECUENCIA_ENCUESTA", "SECUENCIA_P", "ORDEN", "FEX_C"]
)
df = df.merge(right, on="DIRECTORIO", how="left", validate="m:1")

print(df.shape)

(235350, 902)


In [60]:
DF_SIZES["Tamaño Inicial"] = df.shape[0]

In [61]:
print(f"personas: {len(df)}")  # esperado 235350
print(
    f"iguales a 'caracteristicas': {len(df) == len(keys['caracteristicas'])}"
)
print(f"personas repetidas: {df.duplicated(subset=D_SP_ORDEN).sum()}")  # 0
print(
    f"columnas con sufijo: {[c for c in df.columns if c[-2:] in ('_x', '_y')]}"
)
print(f"hogares: {df[D_SP].drop_duplicates().shape[0]}")  # esperado 87060
print(f"viviendas: {df['DIRECTORIO'].nunique()}")  # esperado 86848

personas: 235350
iguales a 'caracteristicas': True
personas repetidas: 0
columnas con sufijo: []
hogares: 87060
viviendas: 86848


In [62]:
# los indicadores de hogar deben ser idénticos para todas las personas
# del hogar: un solo valor distinto por hogar
hogar_vars = ["CANT_PERSONAS_HOGAR", "PERCAPITA", "I_HOGAR", "P5000"]
print(df.groupby(D_SP)[hogar_vars].nunique().max())  # esperado 1 en todas

CANT_PERSONAS_HOGAR    1
PERCAPITA              1
I_HOGAR                1
P5000                  1
dtype: int64


1 significa: en el hogar más desviado de toda la encuesta, esa variable tiene un solo valor distinto. Es decir, no existe ni un hogar donde dos miembros tengan valores diferentes de un indicador de hogar

In [63]:
# la vivienda de dos hogares del contraejemplo de full_analisis.ipynb:
# CANT_PERSONAS_HOGAR se repite dentro del hogar, la edad cambia por persona
cols = [*D_SP_ORDEN, "CANT_PERSONAS_HOGAR", "P6040"]  # P6040 = edad
display(df[df["DIRECTORIO"] == 8370223][cols])

,DIRECTORIO,SECUENCIA_P,ORDEN,CANT_PERSONAS_HOGAR,P6040
1522,8370223,1,1,3,72
1523,8370223,1,2,3,45
1524,8370223,1,3,3,27
1525,8370223,2,1,2,48
1526,8370223,2,2,2,20


Totales de control: **235.350 personas**, 87.060 hogares y 86.848 viviendas,
los mismos del análisis de llaves.

Un `NaN` que venga de un módulo de persona (`educacion`, `tic`,
`fuerza_trabajo`, `trabajo_infantil`, `menores_5`) significa que **a esa
persona no le aplicaba el módulo** por edad, no que la respuesta falte:
`educacion` y `tic` cubren 221.043 personas, `fuerza_trabajo` 195.074,
`trabajo_infantil` 25.969 y `menores_5` 14.307.

El `validate=` de cada cruce es la red de seguridad: `1:1` en los módulos de
persona y `m:1` en hogar y vivienda, así que una llave equivocada levanta un
error en vez de multiplicar filas en silencio.

## Analisis general de tipos de datos

In [64]:
df.dtypes.value_counts()

float64    699
int64      200
str          3
Name: count, dtype: int64

In [65]:
cols_str = df.select_dtypes(include=["object", "string", "str"]).columns
print(cols_str)

Index(['BLOQUE', 'P1_DEPARTAMENTO', 'P1_MUNICIPIO'], dtype='str')


In [66]:
df["BLOQUE"]

0         A00013
1         A00013
2         A00013
3         A00013
4         A00013
           ...  
235345    R34C02
235346    R34C02
235347    R34C02
235348    R34C02
235349    R34C02
Name: BLOQUE, Length: 235350, dtype: str

In [67]:
for k, frame in keys.items():
    if "BLOQUE" in frame.columns:
        print(k)

diseno


`BLOQUE` es la única columna de texto de la tabla final y la única que no es
una respuesta de la encuesta: 

- **No es una predictora**, es una variable de diseño. Al armar la matriz del
  modelo hay que excluirla.
- `BLOQUE` es la partición geográfica en la que el DANE calibra los factores de expansión.
- Sirve para estimaciones por dominio (departamento, cabecera/resto) y, junto
  con `FEX_C`, `ESTRATO2020` y `SEGMENTO`, para calcular errores estándar
  respetando el diseño complejo de la encuesta.

### Usos posibles de `BLOQUE` (aportado por Claude Opus 5.5)

- Para estratificar el split. Separar train y test por BLOQUE y por el target mantiene la representatividad territorial, con SEGMENTO como grupo en el GroupKFold.
- Para evaluar el modelo. Mide el desempeño por bloque y así se ve si falla sistemáticamente en zonas rurales o en ciudades concretas.

In [68]:
df["BLOQUE"].nunique()

94

# Renombrar Columnas
**Asistido por claude Opus 5.5**

In [69]:
import re
import unicodedata

CODE_RE = re.compile(r"^P\d+[A-Za-z0-9]*$")
# F24 trae los sufijos en minúscula (P8587s1) y los CSV en mayúscula
V_UP = {k.upper(): v for k, v in VARIABLES.items()}


def _slug(texto):
    """Pasa la pregunta completa a snake_case sin tildes ni numeración."""
    t = re.sub(r"^\d+[A-Za-z]?\.\s*", "", texto)  # quita "4. ", "12A. "
    t = unicodedata.normalize("NFKD", t)
    t = t.encode("ascii", "ignore").decode()  # quita tildes
    return re.sub(r"[^0-9A-Za-z]+", "_", t).strip("_").lower()


def build_rename(columns):
    """código -> 'CÓDIGO_pregunta'; deja intacto lo que no es un P-código.

    El código se busca con `col.split("_")[0]`, así que la celda se puede
    volver a correr sobre un df que ya trae los nombres largos.
    """
    mapa = {}
    for col in columns:
        codigo = col.split("_")[0]
        info = V_UP.get(codigo.upper())
        if info and CODE_RE.match(codigo):
            pregunta = _slug(info["pregunta"])
            if pregunta:
                mapa[col] = f"{codigo}_{pregunta}"
    return mapa

In [70]:
rename_map = build_rename(df.columns)
df_renamed = df.rename(columns=rename_map)

print(f"renombradas: {len(rename_map)}")  # esperado 629
print(f"intactas: {df.shape[1] - len(rename_map)}")  # esperado 273
print(f"forma: {df.shape}")  # esperado (235350, 902)
print(f"nombres repetidos: {df.columns.duplicated().sum()}")  # esperado 0
print(f"llaves intactas: {all(c in df for c in [*D_SP_ORDEN, 'FEX_C'])}")
display(pd.Series(rename_map).head(8).to_frame("nombre nuevo"))

renombradas: 629
intactas: 273
forma: (235350, 902)
nombres repetidos: 0
llaves intactas: True


,nombre nuevo
P6016,P6016_numero_de_orden_de_la_persona_que_propor...
P1894,P1894_tipo_de_documento_de_identidad
P6020,P6020_sexo_al_nacer
P6034,P6034_cual_es_la_fecha_de_nacimiento_de
P6040,P6040_cuantos_anos_cumplidos_tiene
P6051,P6051_cual_es_el_parentesco_de_con_el_jefe_o_l...
P5502,P5502_actualmente_estado_civil
P6071,P6071_el_la_conyuge_de_vive_en_este_hogar


### Renombrar las columnas con `VARIABLES`

Cada columna que es un código ECV pasa a `CÓDIGO_pregunta`, con el texto
completo de la pregunta normalizado: sin tildes, en `snake_case` y sin la
numeración del cuestionario. Los nombres quedan largos (mediana 63
caracteres, máximo 111), pero no hay ambigüedad: en las familias de preguntas
se conserva tanto el encabezado común como la etiqueta que distingue
(`P1077S1` → `P1077S1_bienes_o_servicios_que_posee_el_hogar_lavadora_de_ropa`).

**El código se conserva al frente a propósito**: el IPM que falta especificar
en `variables_naming.py` indexa por código, y `col.split("_")[0]` lo recupera
sin necesidad de un diccionario inverso. Por eso mismo la celda se puede
volver a correr sobre un `df` ya renombrado.

**Qué queda intacto** (273 columnas):

- Las llaves (`DIRECTORIO`, `SECUENCIA_P`, `ORDEN`) y el bloque de diseño
  (`FEX_C`, `BLOQUE`, `SEGMENTO`, `ESTRATO2020`, `MPIO`, `F2`, `F_ETAPA1`,
  `FRES`), más `PERCAPITA`, `I_HOGAR`, `I_UGASTO`, `I_OU`,
  `CANT_PERSONAS_HOGAR`, `CLASE`, `REGION`, `P1_DEPARTAMENTO` y
  `P1_MUNICIPIO`: ya son nombres legibles y renombrarlos rompería `D_SP` y los
  chequeos de arriba.
- **251 columnas sin etiqueta en `VARIABLES`**, que siguen con su código
  pelado: `salud` 81, `adquisicion` 41, `caracteristicas` 33, `tic` 22,
  `menores_5` 22, `condiciones` 16, `servicios` 12, `educacion` 10,
  `fuerza_trabajo` 9, `vivienda` 4 y `trabajo_infantil` 1.

`adquisicion` no tiene ninguna etiqueta porque *"Adquisición de bienes y
servicios"* no está en `ARCHIVOS`; si esas 41 variables se van a usar, hay que
agregarlas al diccionario.

In [71]:
df

,DIRECTORIO,SECUENCIA_P,ORDEN,P6016,P1894,P6020,P6034,P6040,P6051,P5502,...,P5661S9,P5661S5,P5661S6,P5661S7,P3157,P3157S1,P3157S2,P3157S3,P3157S4,P3158
0,8369550,1,1,1,3,2,1,45,1,5.0,...,1,1,1,1,NaN,2,2,2,2,2
1,8369550,1,2,2,3,1,1,25,7,4.0,...,1,1,1,1,NaN,2,2,2,2,2
2,8369550,1,3,3,2,2,1,16,9,5.0,...,1,1,1,1,NaN,2,2,2,2,2
3,8369550,1,4,4,2,1,1,10,9,5.0,...,1,1,1,1,NaN,2,2,2,2,2
4,8369551,1,1,1,3,2,1,62,1,3.0,...,1,1,2,1,NaN,2,2,2,2,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
235345,8489520,1,2,2,3,2,1,31,2,2.0,...,1,1,1,1,NaN,2,2,2,2,2
235346,8489520,1,3,3,1,2,1,8,3,NaN,...,1,1,1,1,NaN,2,2,2,2,2
235347,8489521,1,1,1,3,1,1,31,1,2.0,...,1,1,1,1,NaN,2,2,2,2,2
235348,8489521,1,2,2,3,2,1,32,2,2.0,...,1,1,1,1,NaN,2,2,2,2,2


# Crear variable objetivo

En esta estapa mas alla de usar el enfoque que uso el paper para el corte se utiliza la forma de realizar el corte replicando el procedimiento reportado por el DANE para la propia ECV del 2025 y se evaluara el corte con el mismo metodo del paper. Cabe resaltar que el DANE usa el mismo enfoque que el paper, pues usa la metodologia de Alkire Foster y el enfoque de Sen

Nota: las variables que se usen para realizar los cortes solo se utilizaran en esta etapa, ademas no se hace sobre el set cortado ya que al ser una etapa de consolidacion cada corte se basa en los datos de un solo hogar, por lo tanto finalmente solo habra una fila por hogar, es decir no hay fuga porque los cortes solo se basan en datos de si mismo.

## Replicar al DANE

Los indicadores, pesos y criterios de privacion que utiliza el DANE en las 4 dimensiones que define (a diferencia de las 5 que el paper utiliza) son:

| Dimensión (peso 0,20) | Indicador | w | z: el hogar está privado si… |
|---|---|---|---|
| Condiciones educativas | Bajo logro educativo | 0,10 | El promedio de años de educación de las personas mayores de 15 años es menor a 9. Los hogares sin mayores de 15 años se consideran privados. |
| Condiciones educativas | Analfabetismo | 0,10 | Al menos una persona mayor de 15 años no sabe leer ni escribir. Los hogares sin mayores de 15 años se consideran privados. |
| Niñez y juventud | Rezago escolar | 0,05 | Al menos un miembro de 7 a 17 años tiene menos años de educación que (edad − 6), por ejemplo, 17 años con menos de 11 años de educación. Los hogares sin miembros de 7 a 17 años no están privados. |
| Niñez y juventud | Inasistencia escolar | 0,05 | Al menos un niño de 6 a 16 años no estudia actualmente. Los hogares sin niños de 6 a 16 años no están privados. |
| Niñez y juventud | Barreras de acceso a servicios de cuidado de la primera infancia | 0,05 | Al menos un niño de 0 a 5 años cumple alguna de estas condiciones: no está asegurado en salud; permanece con su padre o madre en el trabajo, solo en casa o al cuidado de un pariente menor de 18 años; asiste a un jardín o institución educativa pero no recibe alimentación gratis o por un pago simbólico; o, si tiene 5 años, no asiste a una institución educativa. |
| Niñez y juventud | Trabajo infantil | 0,05 | Al menos un miembro de 12 a 17 años está ocupado. |
| Salud | Sin aseguramiento en salud | 0,10 | Al menos un miembro mayor de 5 años no está afiliado a salud. La respuesta "No sabe / No informa" cuenta como no afiliado. |
| Salud | Barreras de acceso a servicios de salud dada una necesidad | 0,10 | Al menos una persona que tuvo un problema de salud en los últimos 30 días no acudió a un médico general, especialista, odontólogo, terapista o institución de salud. Los hogares sin problemas de salud no están privados. |
| Trabajo | Desempleo de larga duración | 0,10 | Al menos una persona de la PEA lleva más de 12 meses buscando trabajo. Los hogares sin PEA o compuestos solo por pensionados se consideran privados. |
| Trabajo | Trabajo informal | 0,10 | No todos los ocupados de la PEA ajustada (sin ocupados menores de 18 años ni desempleados de larga duración) cotizan a pensión o están pensionados. Los hogares sin PEA ajustada están privados; los compuestos solo por pensionados no. |
| Vivienda y servicios públicos | Sin acceso a fuente de agua mejorada | 0,04 | En zona urbana: no tiene acueducto. En zona rural: obtiene el agua para preparar alimentos de pozo sin bomba, agua lluvia, río o manantial, carrotanque, aguatero, o agua embotellada o en bolsa. |
| Vivienda y servicios públicos | Inadecuada eliminación de excretas | 0,04 | En zona urbana: no tiene alcantarillado. En zona rural: tiene inodoro sin conexión, bajamar o no tiene servicio sanitario. |
| Vivienda y servicios públicos | Material inadecuado de pisos | 0,04 | Tiene pisos en tierra. |
| Vivienda y servicios públicos | Material inadecuado de paredes exteriores | 0,04 | En zona urbana: las paredes son de madera burda, tabla, tablón, guadua, otro vegetal, zinc, tela, cartón o desechos, o no tiene paredes. En zona rural: guadua, otro vegetal, zinc, tela, cartón o desechos, o no tiene paredes. |
| Vivienda y servicios públicos | Hacinamiento crítico | 0,04 | En zona urbana: 3 o más personas por cuarto. En zona rural: más de 3 personas por cuarto. |

**Corte de pobreza (k):** un hogar es pobre por IPM si la suma ponderada de sus privaciones es mayor o igual a 5/15 (33,3%).

Note ademas que cada una de las dimensiones reparte su peso de manera igual entre los diferentes indicadores.

### Que variables de las 902 tomar?

A continuacion se listan las variables que se identificaron utiles (con ayuda de Claude Opus 5.5) para replicar este metodo:

| Bloque | Variable (código ECV 2025) | Archivo | Descripción | Indicador(es) que alimenta |
|---|---|---|---|---|
| Identificadores | DIRECTORIO | Todos | Identificador de la vivienda | Llave para unir todos los archivos |
| Identificadores | SECUENCIA_ENCUESTA | Todos | Número de hogar (en archivos de hogar) o de persona (en archivos de personas) | Llave para unir archivos |
| Identificadores | SECUENCIA_P | Todos | Número de hogar dentro de la vivienda (en archivos de personas) | Llave para unir archivos |
| Identificadores | ORDEN | Todos | Número de persona dentro del hogar | Llave para unir archivos |
| Identificadores | FEX_C | Todos | Factor de expansión | Expansión de resultados |
| Personas | P6040 | Características y composición del hogar | Edad en años cumplidos | Todos los filtros por edad |
| Personas | P6020 | Características y composición del hogar | Sexo al nacer | Desagregaciones |
| Personas | P6051 | Características y composición del hogar | Parentesco con el jefe del hogar | Identificación de miembros del hogar |
| Educación | P6160 | Educación | ¿Sabe leer y escribir? | Analfabetismo |
| Educación | P8586 | Educación | ¿Actualmente estudia? | Inasistencia escolar, logro educativo, rezago escolar, primera infancia (niños de 5 años) |
| Educación | P8587 | Educación | Nivel educativo más alto alcanzado | Logro educativo, rezago escolar |
| Educación | P8587S1 | Educación | Grado o año aprobado en ese nivel | Logro educativo, rezago escolar |
| Educación | P1088 | Educación | Nivel en el que está matriculado | Logro educativo, rezago escolar |
| Educación | P1088S1 | Educación | Grado o año que cursa | Logro educativo, rezago escolar |
| Educación | P6180 | Educación | ¿Recibe alimentos en el plantel gratis o por pago simbólico? | Primera infancia (niños de 5 años) |
| Primera infancia | P51 | Atención integral de menores de 5 años | Dónde o con quién permanece la mayor parte del tiempo entre semana | Barreras de acceso a cuidado de la primera infancia |
| Primera infancia | P55 | Atención integral de menores de 5 años | ¿Recibe desayuno o almuerzo en el lugar donde permanece? | Barreras de acceso a cuidado de la primera infancia |
| Primera infancia | P774 | Atención integral de menores de 5 años | ¿Paga por esta alimentación? | Barreras de acceso a cuidado de la primera infancia |
| Salud | P6090 | Salud | Afiliación a seguridad social en salud | Sin aseguramiento en salud, primera infancia |
| Salud | P5665 | Salud | ¿Tuvo un problema de salud en los últimos 30 días sin hospitalización? | Barreras de acceso a salud dada una necesidad |
| Salud | P8563 | Salud | ¿Qué hizo principalmente para tratar ese problema de salud? | Barreras de acceso a salud dada una necesidad |
| Fuerza de trabajo | P6240 | Fuerza de trabajo | Actividad en la que ocupó la mayor parte del tiempo la semana pasada | Trabajo infantil, desempleo de larga duración, trabajo informal |
| Fuerza de trabajo | P6250 | Fuerza de trabajo | ¿Realizó alguna actividad paga por una hora o más? | Trabajo infantil, desempleo de larga duración, trabajo informal |
| Fuerza de trabajo | P6260 | Fuerza de trabajo | ¿Tenía trabajo o negocio aunque no trabajó? | Trabajo infantil, desempleo de larga duración, trabajo informal |
| Fuerza de trabajo | P6270 | Fuerza de trabajo | ¿Trabajó en un negocio sin que le pagaran? | Trabajo infantil, desempleo de larga duración, trabajo informal |
| Fuerza de trabajo | P6280 | Fuerza de trabajo | ¿Hizo diligencias para conseguir trabajo en las últimas 4 semanas? | Desempleo de larga duración (definición de desocupado) |
| Fuerza de trabajo | P6351 | Fuerza de trabajo | ¿Estaba disponible la semana pasada para empezar a trabajar? | Desempleo de larga duración (definición de desocupado) |
| Fuerza de trabajo | P7250 | Fuerza de trabajo | Semanas buscando trabajo (privado si son más de 52) | Desempleo de larga duración |
| Fuerza de trabajo | P6920 | Fuerza de trabajo | ¿Cotiza actualmente a un fondo de pensiones? | Trabajo informal |
| Fuerza de trabajo | P8642 | Fuerza de trabajo | ¿Recibió el mes pasado ingreso por pensión? (condición de pensionado) | Trabajo informal, desempleo de larga duración |
| Vivienda y servicios | CLASE | Datos de la vivienda | Cabecera (urbano) o resto (rural) | Agua, excretas, paredes, hacinamiento |
| Vivienda y servicios | P8520S5 | Datos de la vivienda | ¿Tiene acueducto? | Sin acceso a fuente de agua mejorada (urbano) |
| Vivienda y servicios | P8520S3 | Datos de la vivienda | ¿Tiene alcantarillado? | Inadecuada eliminación de excretas (urbano) |
| Vivienda y servicios | P4015 | Datos de la vivienda | Material predominante de los pisos | Material inadecuado de pisos |
| Vivienda y servicios | P4005 | Datos de la vivienda | Material predominante de las paredes exteriores | Material inadecuado de paredes exteriores |
| Vivienda y servicios | P8530 | Servicios del hogar | Fuente principal del agua para preparar alimentos | Sin acceso a fuente de agua mejorada (rural) |
| Vivienda y servicios | P8526 | Servicios del hogar | Tipo de servicio sanitario | Inadecuada eliminación de excretas (rural) |
| Vivienda y servicios | P5010 | Servicios del hogar | Número de cuartos en que duermen las personas del hogar | Hacinamiento crítico |
| Vivienda y servicios | CANT_PERSONAS_HOGAR | Servicios del hogar | Número de personas que componen el hogar | Hacinamiento crítico |

**Notas:**
- Códigos verificados en el diccionario de la ECV 2025 (microdatos.dane.gov.co, catálogo 905).

- Si P6051 = 1 -> persona es jefe del hogar

- P1088 indica el nivel educativo en el que la persona está matriculada actualmente: preescolar, básica primaria, básica secundaria, media, técnico, tecnológico, universitario, posgrado, etc. P1088S1 indica el grado o año que cursa dentro de ese nivel. Por ejemplo, P1088 = básica secundaria y P1088S1 = 8 significa que la persona está en octavo grado.

**Nota:** Antes de realizar el metodo se filtra el df a aquellos valores que son CLASE=

### Filtrar solo ciudades

In [72]:
df = df.copy()
df.columns = df.columns.str.upper()

dpto = df["P1_DEPARTAMENTO"].str.zfill(2)
mpio = df["P1_MUNICIPIO"].str.zfill(3)
df["DIVIPOLA"] = dpto + mpio

df["CIUDAD"] = pd.to_numeric(df["CLASE"], errors="coerce").eq(1)

df_ciudades = df[df["CIUDAD"]]

In [73]:
df_ciudades

,DIRECTORIO,SECUENCIA_P,ORDEN,P6016,P1894,P6020,P6034,P6040,P6051,P5502,...,P5661S6,P5661S7,P3157,P3157S1,P3157S2,P3157S3,P3157S4,P3158,DIVIPOLA,CIUDAD
0,8369550,1,1,1,3,2,1,45,1,5.0,...,1,1,NaN,2,2,2,2,2,13001,True
1,8369550,1,2,2,3,1,1,25,7,4.0,...,1,1,NaN,2,2,2,2,2,13001,True
2,8369550,1,3,3,2,2,1,16,9,5.0,...,1,1,NaN,2,2,2,2,2,13001,True
3,8369550,1,4,4,2,1,1,10,9,5.0,...,1,1,NaN,2,2,2,2,2,13001,True
4,8369551,1,1,1,3,2,1,62,1,3.0,...,2,1,NaN,2,2,2,2,2,13001,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
235286,8489488,1,1,1,3,2,1,40,1,4.0,...,2,2,NaN,2,2,1,2,1,99624,True
235287,8489488,1,2,2,2,2,1,15,3,5.0,...,2,2,NaN,2,2,1,2,1,99624,True
235288,8489488,1,3,3,2,2,1,11,3,5.0,...,2,2,NaN,2,2,1,2,1,99624,True
235289,8489488,1,4,4,2,2,1,9,3,NaN,...,2,2,NaN,2,2,1,2,1,99624,True


In [74]:
DF_SIZES["Filtrar Ciudades"] = df.shape[0]

In [75]:
cols = list(variables_ipm_dane.keys())
cols.remove("SECUENCIA_ENCUESTA")

df_ciudades[cols]

,DIRECTORIO,SECUENCIA_P,ORDEN,FEX_C,P1_DEPARTAMENTO,P1_MUNICIPIO,CLASE,P6040,P6020,P6051,...,P6920,P8642,P8520S5,P8520S3,P4015,P4005,P8530,P8526,P5010,CANT_PERSONAS_HOGAR
0,8369550,1,1,472.131901,13,1,1,45,2,1,...,2.0,2.0,1,1,4,1,1,1,3,4
1,8369550,1,2,472.131901,13,1,1,25,1,7,...,1.0,2.0,1,1,4,1,1,1,3,4
2,8369550,1,3,472.131901,13,1,1,16,2,9,...,2.0,2.0,1,1,4,1,1,1,3,4
3,8369550,1,4,472.131901,13,1,1,10,1,9,...,NaN,NaN,1,1,4,1,1,1,3,4
4,8369551,1,1,308.854319,13,1,1,62,2,1,...,2.0,2.0,1,1,4,1,1,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
235286,8489488,1,1,6.498987,99,624,1,40,2,1,...,2.0,2.0,1,2,7,1,2,2,2,5
235287,8489488,1,2,6.498987,99,624,1,15,2,3,...,2.0,2.0,1,2,7,1,2,2,2,5
235288,8489488,1,3,6.498987,99,624,1,11,2,3,...,NaN,NaN,1,2,7,1,2,2,2,5
235289,8489488,1,4,6.498987,99,624,1,9,2,3,...,NaN,NaN,1,2,7,1,2,2,2,5


Ya que la dimensionalidad del conjunto de datos es tan alta se aplicara una metodologia alternativa, por cada una de las dimensiones se extraeran 1 o 2 inducadores que en lugar de ser usados en el procedimiento de corte se usaran como caracterisitcas del modelo (refactorizar las entradas para el objeto de corte).

## Como lidear con la dimensionalidad (mas de 900 variables)
Bajo este contexto se plantea eliminar las siguientes variables:

- Analfabetismo
- Trabajo Infantil
- Tiene eps Si/No
- Trabajo Informal
- Sin acceso al agua
- Mala eliminacionde excretas.

Cabe destacar que esto da como un resultado una variable objetivo menos robusta, esto ya que se deriva de menos informacion de los hogares y en aquellas dimensiones que quedan con solo un indicador hogares que solo esten privada en esa dimension y otra ya se consideran multidimensionalmente pobres. Es una falla metodologica que vale la pena destacar.

### Despues de eliminar los indicadores queda asi.

| Dimensión | Peso dimensión | Indicador | Rol | w en el corte | Variables ECV que lo construyen | Nota |
|---|---|---|---|---|---|---|
| Educación | 0,20 | Logro educativo | Corte | 0,2000 | P6040, P8586, P8587, P8587S1, P6211, P1088, P1088S1, P6216 | Único indicador de la dimensión |
| Educación | — | Analfabetismo | Característica | — | P6160 | Revisar R0 frente a logro educativo |
| Niñez y juventud | 0,20 | Rezago escolar | Corte | 0,0667 | P6040, P8586, P8587, P8587S1, P6211, P1088, P1088S1, P6216 | |
| Niñez y juventud | 0,20 | Inasistencia escolar | Corte | 0,0667 | P6040, P8586 | |
| Niñez y juventud | 0,20 | Barreras de acceso a cuidado de primera infancia | Corte | 0,0667 | P6040, P6090, P51, P55, P774, P8586, P6180 | Usa P6090 para menores de 6 años |
| Niñez y juventud | — | Trabajo infantil | Característica | — | P6040, P6240, P6250, P6260, P6270 | P6240 a P6270 también definen la PEA del desempleo de larga duración |
| Salud | 0,20 | Barreras de acceso a salud dada una necesidad | Corte | 0,2000 | P5665, P8563 | Único indicador de la dimensión |
| Salud | — | Sin aseguramiento en salud | Característica | — | P6040, P6090 | Solo personas de 6 años o más; P6090 de menores de 6 años está en el corte |
| Trabajo | 0,20 | Desempleo de larga duración | Corte | 0,2000 | P6040, P6240, P6250, P6260, P6270, P6351, P7250, P6920, P8642 | Único indicador; hogares sin PEA o solo de pensionados quedan privados |
| Trabajo | — | Trabajo informal | Característica | — | P6040, P6240, P6250, P6260, P6270, P6351, P7250, P6920, P8642 | Comparte la PEA y la condición de pensionado con desempleo de larga duración |
| Vivienda y servicios | 0,20 | Material inadecuado de pisos | Corte | 0,0667 | P4015 | |
| Vivienda y servicios | 0,20 | Material inadecuado de paredes exteriores | Corte | 0,0667 | CLASE, P4005 | |
| Vivienda y servicios | 0,20 | Hacinamiento crítico | Corte | 0,0667 | CLASE, CANT_PERSONAS_HOGAR, P5010 | |
| Vivienda y servicios | — | Sin acceso a fuente de agua mejorada | Característica | — | CLASE, P8520S5, P8530 | Sin variables compartidas con el corte |
| Vivienda y servicios | — | Inadecuada eliminación de excretas | Característica | — | CLASE, P8520S3, P8526 | Sin variables compartidas con el corte |

**Corte de pobreza (k):** 5/15 (33,3%) sobre los 9 indicadores que quedan en el corte.

In [76]:
variables_liberadas = [
    "P6160",  # ¿Sabe leer y escribir? (analfabetismo)
    "P8520S5",  # ¿Tiene acueducto? (agua)
    "P8530",  # Fuente del agua para preparar alimentos (agua)
    "P8520S3",  # ¿Tiene alcantarillado? (excretas)
    "P8526",  # Tipo de servicio sanitario (excretas)
]

## Desarrollar doble corte usando la metodologia Alkire-Foster

Se aplica la adaptacion anteriormente descrita y se compara la robustes de esta frente a la robustes de la metodologia del dane, asimismo se procede a realizar la separacion en train, validation y test (esto antes de evaluar la robustes con el fin de evitar el data leakage). El metodo de doble corte se implemento como una clase compatible con el metodo Pipeline de sklearn para ser usado de manera mas robusta y confiable, cabe resaltar que el codigo se realizo con la asistencia de Claude opus 5.5.




In [77]:
df

,DIRECTORIO,SECUENCIA_P,ORDEN,P6016,P1894,P6020,P6034,P6040,P6051,P5502,...,P5661S6,P5661S7,P3157,P3157S1,P3157S2,P3157S3,P3157S4,P3158,DIVIPOLA,CIUDAD
0,8369550,1,1,1,3,2,1,45,1,5.0,...,1,1,NaN,2,2,2,2,2,13001,True
1,8369550,1,2,2,3,1,1,25,7,4.0,...,1,1,NaN,2,2,2,2,2,13001,True
2,8369550,1,3,3,2,2,1,16,9,5.0,...,1,1,NaN,2,2,2,2,2,13001,True
3,8369550,1,4,4,2,1,1,10,9,5.0,...,1,1,NaN,2,2,2,2,2,13001,True
4,8369551,1,1,1,3,2,1,62,1,3.0,...,2,1,NaN,2,2,2,2,2,13001,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
235345,8489520,1,2,2,3,2,1,31,2,2.0,...,1,1,NaN,2,2,2,2,2,99524,False
235346,8489520,1,3,3,1,2,1,8,3,NaN,...,1,1,NaN,2,2,2,2,2,99524,False
235347,8489521,1,1,1,3,1,1,31,1,2.0,...,1,1,NaN,2,2,2,2,2,99524,False
235348,8489521,1,2,2,3,2,1,32,2,2.0,...,1,1,NaN,2,2,2,2,2,99524,False


In [78]:
ipm = IPM(caracteristicas=INDICADORES_ELIMINADOS)
ipm.fit(df_ciudades)


,caracteristicas,"['analfabetismo', 'trabajo_infantil', ...]"
,indicadores,None
,w,None
,k,None
,codigos,None
,caracteristicas_nan,False
,claves_hogar,"('DIRECTORIO', ...)"
,col_peso,'FEX_C'
,col_clase,'CLASE'
Name,Type,Value
caracteristicas_,list,"['an...mo', 'tr...il', 'si...to', 'tr...al', ...]"


In [79]:
ipm.w_

{'logro_educativo': np.float64(0.2),
 'rezago_escolar': np.float64(0.06666666666666667),
 'inasistencia_escolar': np.float64(0.06666666666666667),
 'primera_infancia': np.float64(0.06666666666666667),
 'barreras_salud': np.float64(0.2),
 'desempleo_larga': np.float64(0.2),
 'pisos': np.float64(0.06666666666666667),
 'paredes': np.float64(0.06666666666666667),
 'hacinamiento': np.float64(0.06666666666666667)}

In [80]:
sum(list(ipm.w_.values()))

np.float64(1.0)

In [81]:
ipm.caracteristicas_ # Los datos que se usaran en el modelo

['analfabetismo',
 'trabajo_infantil',
 'sin_aseguramiento',
 'trabajo_informal',
 'agua',
 'excretas']

In [82]:
df_ciudad_w_target = ipm.transform(df_ciudades)
df_ciudad_w_target

,DIRECTORIO,SECUENCIA_P,PESO,logro_educativo,rezago_escolar,inasistencia_escolar,primera_infancia,barreras_salud,desempleo_larga,pisos,paredes,hacinamiento,CAR_analfabetismo,CAR_trabajo_infantil,CAR_sin_aseguramiento,CAR_trabajo_informal,CAR_agua,CAR_excretas,C,POBRE
0,8369550,1,1888.527605,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0.066667,0
1,8369551,1,308.854319,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0.200000,0
2,8369552,1,2142.581195,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0.066667,0
3,8369553,1,806.929968,1,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0.400000,1
4,8369554,1,707.898884,1,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0.400000,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44346,8489484,1,12.997974,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0.000000,0
44347,8489485,1,19.496961,0,0,0,1,0,0,0,0,1,0,0,0,1,0,1,0.133333,0
44348,8489486,1,12.997974,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0.000000,0
44349,8489487,1,51.343437,0,1,0,0,0,0,0,0,0,0,0,0,1,0,1,0.066667,0


In [83]:
DF_SIZES["Filtrar Ciudades (Hogares)"] = df_ciudad_w_target.shape[0]

In [84]:
DF_SIZES

{'Tamaño Inicial': 235350,
 'Filtrar Ciudades': 235350,
 'Filtrar Ciudades (Hogares)': 44351}

## Eliminar columnas de las que se deriva el calculo de la target, EDA caracteristicas, evaluar VIF de las caracteristicas y ver su correlación con el objetivo.

Tambien se evaluan las correlaciones contra los indicadores que si se usaron para evitar fuga o se consientes de la limitación tecnica.

In [91]:
df_ciudad_w_target_only_features = df_ciudad_w_target[
    CARACTERISTICAS_NUEVAS + ["POBRE"]
].copy()

In [92]:
df_ciudad_w_target_only_features.describe()

,CAR_analfabetismo,CAR_trabajo_infantil,CAR_sin_aseguramiento,CAR_agua,CAR_excretas,POBRE
count,44351.000000,44351.000000,44351.000000,44351.000000,44351.000000,44351.000000
mean,0.064170,0.010124,0.043246,0.076999,0.155735,0.128340
std,0.245058,0.100108,0.203413,0.266593,0.362608,0.334472
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [93]:
for col in df_ciudad_w_target_only_features.columns:
    print(f"col: {col}")
    display(df_ciudad_w_target_only_features[col].unique())

col: CAR_analfabetismo


array([0, 1])

col: CAR_trabajo_infantil


array([0, 1])

col: CAR_sin_aseguramiento


...

array([0, 1])

col: CAR_excretas


array([0, 1])

col: POBRE


array([0, 1])